# Notebook 2 - Recommendation Models

building 3 types of recommenders - collaborative filtering, content based and popularity based

In [ ]:
import pandas as pd
import numpy as np
import warnings
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from surprise import Dataset, Reader, SVD
from surprise.model_selection import cross_validate

warnings.filterwarnings('ignore')
np.random.seed(42)

# load data
ratings = pd.read_csv('data/ratings.csv', nrows=100000, dtype={
    'userId':  'int32',
    'movieId': 'int32',
    'rating':  'float32'
})
ratings.drop(columns=['timestamp'], errors='ignore', inplace=True)
movies  = pd.read_csv('data/movies.csv', dtype={
    'movieId': 'int32'
})
df = pd.merge(ratings, movies, on='movieId', how='left')

print(f'Ratings: {ratings.shape}  Movies: {movies.shape}  Merged: {df.shape}')
print(df.head())

---
## Collaborative Filtering

using SVD here, it basically finds patterns in how users rate movies and predicts what they might like

In [ ]:
# Load ratings into Surprise format
reader = Reader(rating_scale=(0.5, 5.0))
surprise_data = Dataset.load_from_df(ratings[['userId', 'movieId', 'rating']], reader)

# train SVD, keeping params small so it doesnt take forever
svd_model = SVD(n_factors=20, n_epochs=10, random_state=42)
print('Running 5-fold cross-validation... (~1 min)')
cv_results = cross_validate(svd_model, surprise_data, measures=['RMSE', 'MAE'], cv=2, verbose=False)

print(f'\nSVD Results:')
print(f'  RMSE : {cv_results["test_rmse"].mean():.4f}  (lower is better)')
print(f'  MAE  : {cv_results["test_mae"].mean():.4f}  (lower is better)')

# train on full dataset for making recommendations
trainset = surprise_data.build_full_trainset()
svd_model.fit(trainset)
print('\nSVD trained on full dataset')

In [ ]:
def get_cf_recommendations(user_id, n=10):
    """
    Get top-N movie recommendations for a user using SVD.
    Predicts ratings for all unrated movies and returns the top N.
    """
    all_movie_ids  = ratings['movieId'].unique()
    rated_by_user  = ratings[ratings['userId'] == user_id]['movieId'].tolist()
    unrated_movies = [m for m in all_movie_ids if m not in rated_by_user]

    predictions = [(mid, round(svd_model.predict(user_id, mid).est, 2))
                   for mid in unrated_movies]
    predictions.sort(key=lambda x: x[1], reverse=True)

    result = pd.DataFrame(predictions[:n], columns=['movieId', 'predicted_rating'])
    result = pd.merge(result, movies, on='movieId', how='left')
    return result[['title', 'genres', 'predicted_rating']]


print('Top 10 CF Recommendations for User 1:')
print(get_cf_recommendations(user_id=1, n=10).to_string(index=False))

---
## Content-Based Filtering

using tfidf on genres and cosine similarity to find similar movies, took me a while to figure out the memory issue with the full matrix

In [ ]:
# Replace pipe with space so TF-IDF treats each genre as a separate word
movies['genres_clean'] = movies['genres'].str.replace('|', ' ', regex=False)

tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(movies['genres_clean'])

# not computing the full matrix, doing it per query instead
title_to_idx = pd.Series(movies.index, index=movies['title']).to_dict()

print(f'TF-IDF matrix: {tfidf_matrix.shape}')
print('Cosine similarity: computed per query (memory-safe)')

In [ ]:
def get_content_recommendations(movie_title, n=10):
    """
    Return top-N genre-similar movies for a given title.
    OPTIMIZED: computes similarity for one query row only.
    Not the full N x N matrix — saves huge amount of RAM.
    """
    from sklearn.metrics.pairwise import cosine_similarity as cos_sim

    if movie_title not in title_to_idx:
        matches = [t for t in title_to_idx 
                   if movie_title.lower() in t.lower()]
        if not matches:
            print(f'Movie "{movie_title}" not found.')
            return pd.DataFrame()
        movie_title = matches[0]
        print(f'Using closest match: "{movie_title}"')

    idx       = title_to_idx[movie_title]
    query_vec = tfidf_matrix[idx]                   # single sparse row
    sim_row   = cos_sim(query_vec, tfidf_matrix)[0] # shape (n_movies,)

    sim_scores = list(enumerate(sim_row))
    sim_scores.sort(key=lambda x: x[1], reverse=True)
    top_n = [s for s in sim_scores if s[0] != idx][:n]

    top_indices = [i[0] for i in top_n]
    scores      = [round(i[1], 4) for i in top_n]

    result = movies.iloc[top_indices][['title', 'genres']].copy()
    result['similarity_score'] = scores
    return result.reset_index(drop=True)


print('Top 10 Content Recommendations for "Toy Story (1995)":')
print(get_content_recommendations('Toy Story (1995)', n=10).to_string(index=False))

---
## Popularity-Based

using the IMDB weighted rating formula so movies with very few votes dont rank at the top

`score = (v/(v+m)) × R + (m/(v+m)) × C`

In [ ]:
# Compute stats per movie
movie_stats = df.groupby(['movieId', 'title', 'genres'])['rating'].agg(
    num_votes='count', avg_rating='mean').reset_index()

C = df['rating'].mean()                           # global mean rating
m = movie_stats['num_votes'].quantile(0.50)       # 50th-percentile vote threshold

print(f'Global mean rating (C) : {C:.4f}')
print(f'Vote threshold (m)     : {m:.0f}')

# apply weighted formula
movie_stats['weighted_score'] = (
    (movie_stats['num_votes'] / (movie_stats['num_votes'] + m)) * movie_stats['avg_rating'] +
    (m / (movie_stats['num_votes'] + m)) * C
).round(4)


def get_popular_movies(genre=None, n=10):
    """
    Return top-N movies by weighted score.
    If genre is given, filter to only that genre first.
    """
    result = movie_stats.copy()
    if genre:
        result = result[result['genres'].str.contains(genre, case=False, na=False)]
        if result.empty:
            print(f'No movies found for genre: {genre}')
            return pd.DataFrame()
    result = result.sort_values('weighted_score', ascending=False)
    return result[['title', 'genres', 'num_votes', 'avg_rating', 'weighted_score']].head(n).reset_index(drop=True)


print('\n=== Top 10 Popular Movies (all genres) ===')
print(get_popular_movies(n=10).to_string(index=False))

print('\n=== Top 10 Popular Action Movies ===')
print(get_popular_movies(genre='Action', n=10).to_string(index=False))

print('\n=== Top 10 Popular Animation Movies ===')
print(get_popular_movies(genre='Animation', n=10).to_string(index=False))

print('\ndone')